# 1. Inspect Raw Data

Goal: understand the Rossmann Store Sales dataset before deciding how to build per-store daily series - date range, missingness, closed-store days, and what each column means. Corresponds to step 1 of the workflow in `CLAUDE.md`.

In [ ]:
import sys
from pathlib import Path

SRC = Path.cwd().parent / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd

from data import load_train, load_store, missingness_report

pd.set_option("display.max_columns", 50)

## Load the raw files

In [ ]:
train = load_train()
store = load_store()
print(f"train.csv: {train.shape[0]:,} rows, {train.shape[1]} columns")
print(f"store.csv: {store.shape[0]:,} rows, {store.shape[1]} columns")
train.head()

In [ ]:
store.head()

## Date range and coverage

In [ ]:
print("Date range:", train["Date"].min().date(), "to", train["Date"].max().date())
print("Number of distinct stores:", train["Store"].nunique())
print("Calendar days spanned:", (train["Date"].max() - train["Date"].min()).days + 1)

## Missingness

In [ ]:
print("train.csv missingness:")
display(missingness_report(train))
print("\nstore.csv missingness:")
display(missingness_report(store))

`CompetitionDistance` is missing for a handful of stores (no competitor on record). The `Promo2Since*` and `PromoInterval` columns are missing exactly when `Promo2 == 0` (no continuous promotion running) - this is structural missingness, not a data-quality problem, and needs no imputation.

## Closed-store days

In [ ]:
closed_share = 1 - train["Open"].mean()
print(f"Share of store-days closed overall: {closed_share:.1%}")

train.groupby("DayOfWeek")["Open"].mean().rename("share_open")

Most stores close on Sundays (`DayOfWeek == 7`), plus scattered closures for holidays/refurbishment. `Sales` is 0 whenever `Open == 0` by construction - handled explicitly in `src/timeseries.py`, which drops closed days rather than modeling them as zero demand (see that module's docstring for the full reasoning).

## Column reference

**train.csv**
- `Store` - store id (joins to store.csv)
- `DayOfWeek` - 1 (Monday) .. 7 (Sunday)
- `Date` - calendar date
- `Sales` - turnover for the day (target variable)
- `Customers` - number of customers that day
- `Open` - 0 = closed, 1 = open
- `Promo` - 1 if a daily promotion was running
- `StateHoliday` - none / public / easter / christmas
- `SchoolHoliday` - 1 if schools were closed that day (affects footfall)

**store.csv**
- `StoreType` - store model/format, categories a-d
- `Assortment` - product range breadth, a/b/c
- `CompetitionDistance` - meters to nearest competitor
- `CompetitionOpenSince{Month,Year}` - when that competitor opened
- `Promo2` - 1 if the store runs a continuous (not just daily) promotion
- `Promo2Since{Week,Year}` - when the continuous promotion started
- `PromoInterval` - which months the continuous promotion re-runs in

In [ ]:
train[["Sales", "Customers"]].describe()